# APS 2026 Workshop: Python for Detection of Downy Mildew using Hyperspectral sensing

**Case study:** top-canopy hyperspectral reflectance for symptomatic vs asymptomatic leaves infected with grapevine downy mildew.

By the end of this notebook, participants will be able to:

1. Load and inspect hyperspectral reflectance data.
2. Filter the dataset to top-canopy samples only.
3. Plot mean spectral responses by symptom class.
4. Calculate a few vegetation indices that plant pathologists can interpret biologically.
5. Use principal component analysis (PCA) to reduce high-dimensional spectra and visualize sample separation.
6. Train a random forest model using the first four principal components.
7. Evaluate the model with confusion matrices, accuracy, balanced accuracy, recall, precision, F1 score, and AUC.
8. Use AI tools responsibly as coding assistants and analysis reviewers.

> This notebook is designed for a 45 minute workshop. 


## 0. Setup

This notebook uses common Python data science packages. If a package is missing, uncomment and run the install line below.

In [ ]:
# Optional, only if needed:
# This installs the Python packages used in the notebook.
# Run this line only if an import below fails because a package is missing.
# %pip install pandas numpy matplotlib seaborn scikit-learn

# pathlib helps us work with file and folder paths in a way that works on Mac and Windows.
from pathlib import Path

# re is used for regular expressions, which let us identify wavelength columns like X350, X351, etc.
import re

# warnings lets us hide non-critical warning messages so beginners can focus on the main outputs.
import warnings

# numpy is used for fast numeric calculations and arrays.
import numpy as np

# pandas is used for spreadsheet-like tables: reading CSVs, filtering rows, and calculating summaries.
import pandas as pd

# matplotlib and seaborn are used to make plots.
import matplotlib.pyplot as plt
import seaborn as sns

# train_test_split separates data into training and testing sets for model evaluation.
from sklearn.model_selection import train_test_split

# StandardScaler standardizes each wavelength so PCA is not dominated by variables with larger numeric ranges.
from sklearn.preprocessing import StandardScaler

# PCA reduces many wavelength columns into a smaller number of principal components.
from sklearn.decomposition import PCA

# RandomForestClassifier trains a machine-learning classifier made of many decision trees.
from sklearn.ensemble import RandomForestClassifier

# These metrics help us evaluate classification performance from several angles.
from sklearn.metrics import (
    accuracy_score,              # overall fraction of correct predictions
    balanced_accuracy_score,     # average recall across classes; useful when classes are imbalanced
    precision_score,             # of predicted symptomatic samples, how many were truly symptomatic
    recall_score,                # of true symptomatic samples, how many the model found
    f1_score,                    # balance between precision and recall
    roc_auc_score,               # how well predicted probabilities rank the two classes
    confusion_matrix,            # table of correct and incorrect predictions by class
    ConfusionMatrixDisplay,      # helper for plotting a confusion matrix
)

# Hide a common class of warnings that are not important for this beginner workshop.
warnings.filterwarnings("ignore", category=UserWarning)

# Set a clean default plotting style for all seaborn/matplotlib figures.
sns.set_theme(style="whitegrid")

# This is the expected workshop folder name.
# The code below tries to find this folder automatically so paths work on different computers.
WORKSHOP_FOLDER_NAME = "Python for Detection of Downy Mildew using Hyperspectral sensing"

# If Jupyter was launched from inside the workshop folder, use the current folder.
if Path.cwd().name == WORKSHOP_FOLDER_NAME:
    WORKSHOP_DIR = Path.cwd()

# If Jupyter was launched from the parent folder, move into the workshop folder.
elif (Path.cwd() / WORKSHOP_FOLDER_NAME).exists():
    WORKSHOP_DIR = Path.cwd() / WORKSHOP_FOLDER_NAME

# Otherwise, use the current folder as a fallback.
# Beginners can manually edit WORKSHOP_DIR here if the printed path below is wrong.
else:
    # If Jupyter opens in a different folder on your computer, change the next line to your downloaded workshop folder.
    # Example: WORKSHOP_DIR = Path.home() / "Downloads" / WORKSHOP_FOLDER_NAME
    WORKSHOP_DIR = Path.cwd()

# Create an outputs folder for figures and tables made by the notebook.
# exist_ok=True means "do not give an error if the folder already exists."
OUTPUT_DIR = WORKSHOP_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Print paths so participants can confirm the notebook is saving files in the expected place.
print(f"Workshop folder: {WORKSHOP_DIR}")
print(f"Outputs will be saved to: {OUTPUT_DIR}")
print("If the workshop folder above is not correct, update WORKSHOP_DIR in this cell.")


## 1. Load the Dataset

This notebook expects a table where each row is a sample and each wavelength is a column. The wavelength columns are named like `X350`, `X351`, ..., `X2500`.

Update `DATA_PATH` below.

In [ ]:
# The CSV file should be stored in the same workshop folder as this notebook.
# DATA_PATH is the full path to the dataset file that pandas will read.
DATA_PATH = WORKSHOP_DIR / "hyperspectral_data.csv"

# If your CSV file has a different name or is stored in a different folder on your computer, change the line above.
# Example: DATA_PATH = Path.home() / "Downloads" / WORKSHOP_FOLDER_NAME / "your_dataset_name.csv"

# This check stops the notebook early with a clear message if the dataset cannot be found.
# That is easier for beginners than getting a long pandas error later.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the dataset at {DATA_PATH}. Update DATA_PATH to match the CSV file on your computer."
    )

# Read the CSV file into a pandas DataFrame.
# A DataFrame is like a spreadsheet: rows are samples and columns are metadata or wavelengths.
df = pd.read_csv(DATA_PATH)

# Print the shape as (number of rows, number of columns) so we can quickly check the file loaded correctly.
print(df.shape)

# Show the first five rows to inspect column names and example values.
df.head()


## 2. Identify Wavelength Columns and Filter to Top Canopy

We will use only top-canopy spectra and label samples as `asymptomatic` or `symptomatic`.

In [ ]:
# This helper function extracts the wavelength number from column names like X350 or X2500.
# If a column is not a wavelength column, it returns None.
def wavelength_from_column(col):
    match = re.fullmatch(r"X(\d+)", str(col))
    return int(match.group(1)) if match else None

# Keep only columns whose names look like wavelength bands.
# These are the predictor variables used for spectral plots, PCA, and modeling.
wavelength_cols = [col for col in df.columns if wavelength_from_column(col) is not None]

# Sort wavelength columns numerically so the spectra plot from short to long wavelengths.
wavelength_cols = sorted(wavelength_cols, key=wavelength_from_column)

# Store the numeric wavelengths in a numpy array for plotting on the x-axis.
wavelengths = np.array([wavelength_from_column(col) for col in wavelength_cols])

# Print a quick summary so participants can confirm the spectral range.
print(f"Number of wavelength bands: {len(wavelength_cols)}")
print(f"Wavelength range: {wavelengths.min()}-{wavelengths.max()} nm")

# These metadata columns are required for this workshop workflow.
# side tells us top vs bottom canopy; symptoms gives the disease label.
required_cols = {"side", "symptoms"}
missing = required_cols - set(df.columns)

# Stop with a clear error if required columns are missing.
if missing:
    raise ValueError(f"Missing required metadata columns: {missing}")

# Filter to top-canopy samples only, then keep only the two symptom classes used in this workshop.
# .copy() makes a separate DataFrame so later edits do not affect the original df.
top = df.loc[
    df["side"].astype(str).str.lower().eq("top")
    & df["symptoms"].isin(["asymptomatic", "symptomatic"])
].copy()

# Check how many top-canopy samples remain and how many are in each symptom group.
print(f"Top-canopy samples: {len(top)}")
display(top["symptoms"].value_counts())


## 3. Quick Data Check

Before modeling, check that the spectral matrix is numeric and that missing values are handled. Here we fill missing values using the median reflectance for each wavelength.

In [ ]:
# X is the spectral predictor matrix: one row per sample and one column per wavelength.
# apply(pd.to_numeric) makes sure all wavelength values are treated as numbers.
X = top[wavelength_cols].apply(pd.to_numeric, errors="coerce")

# y is the symptom label for each sample.
y = top["symptoms"].copy()

# Calculate the overall fraction of missing values in the spectral data.
# This is a quick data-quality check before analysis.
missing_fraction = X.isna().mean().mean()
print(f"Overall missing-value fraction in spectral matrix: {missing_fraction:.4f}")

# Fill missing wavelength values using the median value for that wavelength.
# This is a simple beginner-friendly imputation method so PCA/modeling can run.
X = X.fillna(X.median())

# Keep this label order for plots and interpretation.
label_order = ["asymptomatic", "symptomatic"]

# Convert text labels into 0/1 labels for machine learning.
# 0 = asymptomatic, 1 = symptomatic.
y_binary = y.map({"asymptomatic": 0, "symptomatic": 1})

# Print the final spectral matrix size and the number of samples in each class.
print(X.shape)
display(y.value_counts())


## 4. Plot Mean Spectra by Symptom Status

This plot helps to visualize hyperspectral data as interpretable curves rather than just a large table.


In [ ]:
# Create one figure and one plotting area.
# figsize controls the width and height of the figure in inches.
fig, ax = plt.subplots(figsize=(10, 5.5))

# Loop over the two symptom groups so both mean spectra are plotted on the same axes.
for status, color in [("asymptomatic", "#2f7d32"), ("symptomatic", "#b23a48")]:
    # Select only samples from the current symptom group.
    group = X.loc[y == status]

    # Calculate the mean reflectance at each wavelength for this group.
    mean = group.mean(axis=0).to_numpy()

    # Calculate the standard error at each wavelength to show uncertainty around the mean.
    se = group.sem(axis=0).to_numpy()

    # Plot the group mean spectrum.
    ax.plot(wavelengths, mean, label=f"{status} (n={len(group)})", color=color, linewidth=2)

    # Add a shaded band of +/- 1 standard error around the mean spectrum.
    ax.fill_between(wavelengths, mean - se, mean + se, color=color, alpha=0.18, linewidth=0)

# Add labels and a legend so the figure is interpretable without reading the code.
ax.set_title("Top-canopy mean reflectance by symptom status")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance")
ax.legend(frameon=False)

# Adjust spacing so labels do not overlap.
fig.tight_layout()

# Save the figure to the outputs folder at high resolution.
fig_path = OUTPUT_DIR / "mean_spectra_top_canopy_by_symptom.png"
fig.savefig(fig_path, dpi=200)

# Display the saved file path as the cell output.
fig_path


## 5. Calculate Vegetation Indices

Vegetation indices reduce selected wavelengths into interpretable ratios. They are not a replacement for full-spectrum modeling, but they are useful because they connect spectra to plant physiology.

We will calculate:

- **NDVI**: canopy vigor / greenness, using red and NIR bands.
- **NDRE**: red-edge chlorophyll/stress signal.
- **PRI**: photosynthetic reflectance index, often linked to xanthophyll-cycle activity.
- **PSRI**: pigment senescence/stress index.

Because exact wavelengths may differ by sensor, the helper function below uses the nearest available band.


In [ ]:
# This function finds the column closest to a target wavelength.
# We use it because sensors do not always have exactly 531, 570, 670, 720, 750, or 800 nm bands.
def nearest_band(target_nm):
    idx = np.abs(wavelengths - target_nm).argmin()
    return wavelength_cols[idx], wavelengths[idx]


# This function returns the reflectance values for the band nearest to the requested wavelength.
def band_values(data, target_nm):
    band, actual_nm = nearest_band(target_nm)
    return data[band], actual_nm


# Many vegetation indices are normalized differences: (high - low) / (high + low).
# This helper reduces repeated code and records the actual wavelengths used.
def normalized_difference(data, high_nm, low_nm):
    high, high_actual = band_values(data, high_nm)
    low, low_actual = band_values(data, low_nm)
    return (high - low) / (high + low), high_actual, low_actual

# Create a new table to store vegetation indices for each sample.
indices = pd.DataFrame(index=top.index)

# NDVI often captures greenness/vigor using near-infrared and red reflectance.
indices["NDVI"], ndvi_nir, ndvi_red = normalized_difference(X, 800, 670)

# NDRE uses a red-edge band and can be sensitive to chlorophyll/stress changes.
indices["NDRE"], ndre_nir, ndre_re = normalized_difference(X, 800, 720)

# PRI is often used as a photosynthetic stress or xanthophyll-cycle index.
r531, pri_531 = band_values(X, 531)
r570, pri_570 = band_values(X, 570)
indices["PRI"] = (r531 - r570) / (r531 + r570)

# PSRI is often associated with pigment senescence or stress.
r680, psri_680 = band_values(X, 680)
r500, psri_500 = band_values(X, 500)
r750, psri_750 = band_values(X, 750)
indices["PSRI"] = (r680 - r500) / r750

# Add symptom labels so the index table can be grouped and plotted by disease status.
indices["symptoms"] = y.values

# Remove infinite or missing values that can occur if an index divides by zero.
indices = indices.replace([np.inf, -np.inf], np.nan).dropna()

# Print the actual nearest wavelengths used so the analysis is transparent.
print("Nearest bands used:")
print(f"NDVI: NIR {ndvi_nir} nm, red {ndvi_red} nm")
print(f"NDRE: NIR {ndre_nir} nm, red-edge {ndre_re} nm")
print(f"PRI: {pri_531} nm and {pri_570} nm")
print(f"PSRI: {psri_680} nm, {psri_500} nm, {psri_750} nm")

# Save the vegetation-index table for later use or reporting.
indices.to_csv(OUTPUT_DIR / "vegetation_indices_top_canopy.csv", index=False)

# Show group means as a quick first look at whether indices differ by symptom status.
display(indices.groupby("symptoms").mean(numeric_only=True))


In [ ]:
# AI Prompt: Ask AI to explain unfamiliar Python code
#
# This helper-function block is a good place to pause and ask AI for a guided explanation.
#
# Try this prompt:
# "Explain this Python code line by line for a plant pathologist who is new to hyperspectral data analysis.
# Also explain why the notebook uses the nearest available wavelength bands for NDVI, NDRE, PRI, and PSRI."


## 6. Plot Vegetation Indices

Boxplots make it easy to discuss whether common spectral indices differ between asymptomatic and symptomatic samples.


In [ ]:
# Convert the vegetation-index table from wide format to long format.
# Wide format has one column per index; long format is easier for seaborn boxplots.
plot_df = indices.melt(
    id_vars="symptoms",
    value_vars=["NDVI", "NDRE", "PRI", "PSRI"],
    var_name="index",
    value_name="value",
)

# Create a figure for the vegetation-index boxplots.
fig, ax = plt.subplots(figsize=(9, 5.5))

# Boxplots summarize the distribution of each index for each symptom group.
sns.boxplot(
    data=plot_df,
    x="index",
    y="value",
    hue="symptoms",
    hue_order=["asymptomatic", "symptomatic"],
    palette={"asymptomatic": "#2f7d32", "symptomatic": "#b23a48"},
    ax=ax,
)

# Stripplots add individual sample points on top of the boxes.
# This helps students see sample size and overlap between groups.
sns.stripplot(
    data=plot_df,
    x="index",
    y="value",
    hue="symptoms",
    hue_order=["asymptomatic", "symptomatic"],
    dodge=True,
    palette={"asymptomatic": "#2f7d32", "symptomatic": "#b23a48"},
    alpha=0.35,
    size=3,
    ax=ax,
)

# seaborn creates duplicate legend entries because we used both boxplot and stripplot.
# Keep only the first two entries: asymptomatic and symptomatic.
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Symptom status", frameon=False)

# Add clear plot labels.
ax.set_title("Top-canopy vegetation indices by symptom status")
ax.set_xlabel("")
ax.set_ylabel("Index value")
fig.tight_layout()

# Save the plot to the outputs folder.
fig_path = OUTPUT_DIR / "vegetation_indices_by_symptom.png"
fig.savefig(fig_path, dpi=200)
fig_path


## 7. PCA for Dimensionality Reduction and Visualization

Principal component analysis (PCA) compresses hundreds or thousands of correlated wavelengths into a smaller number of synthetic axes called principal components. Unlike supervised classification models, PCA does **not** use symptom labels. It shows major spectral variation, which may or may not align with disease status.

This PCA is for visualization and teaching. In the modeling section below, PCA is fitted on the training set only to avoid leakage.


In [ ]:
# PCA is sensitive to scale, so first standardize each wavelength.
# This makes each wavelength have mean 0 and standard deviation 1.
scaler_viz = StandardScaler()
X_scaled_viz = scaler_viz.fit_transform(X)

# Fit PCA for visualization using the full top-canopy dataset.
# n_components=10 keeps the first 10 principal components.
pca_viz = PCA(n_components=10, random_state=42)
pca_scores = pca_viz.fit_transform(X_scaled_viz)

# Store the first four PC scores in a table for plotting and saving.
pca_scores_df = pd.DataFrame(
    pca_scores[:, :4],
    columns=["PC1", "PC2", "PC3", "PC4"],
    index=top.index,
)

# Add symptom labels so points can be colored by disease status.
pca_scores_df["symptoms"] = y.values

# Make a table showing how much spectral variation each PC explains.
variance_df = pd.DataFrame({
    "PC": [f"PC{i}" for i in range(1, 11)],
    "variance_explained": pca_viz.explained_variance_ratio_,
    "cumulative_variance_explained": np.cumsum(pca_viz.explained_variance_ratio_),
})

# Save PCA scores and variance explained for later reference.
pca_scores_df.to_csv(OUTPUT_DIR / "pca_scores_top_canopy.csv", index=False)
variance_df.to_csv(OUTPUT_DIR / "pca_variance_explained_top_canopy.csv", index=False)

# Display the first few PCs so participants can see how much variance is captured.
display(variance_df.head())


In [ ]:
# Create a two-panel figure: PCA scatterplot on the left, variance explained on the right.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot PC1 vs PC2. Each point is one top-canopy sample.
sns.scatterplot(
    data=pca_scores_df,
    x="PC1",
    y="PC2",
    hue="symptoms",
    hue_order=["asymptomatic", "symptomatic"],
    palette={"asymptomatic": "#2f7d32", "symptomatic": "#b23a48"},
    s=65,
    alpha=0.85,
    edgecolor="white",
    linewidth=0.5,
    ax=axes[0],
)

# Add zero reference lines to make the PCA axes easier to read.
axes[0].axhline(0, color="0.75", linewidth=0.8)
axes[0].axvline(0, color="0.75", linewidth=0.8)

# Label axes with the percent variance explained by PC1 and PC2.
axes[0].set_title("PCA scores by symptom status")
axes[0].set_xlabel(f"PC1 ({pca_viz.explained_variance_ratio_[0] * 100:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({pca_viz.explained_variance_ratio_[1] * 100:.1f}% variance)")
axes[0].legend(title="Symptom status", frameon=False)

# Plot the variance explained by the first 10 PCs.
sns.barplot(
    data=variance_df.head(10),
    x="PC",
    y="variance_explained",
    color="#4c78a8",
    ax=axes[1],
)

# Add cumulative variance so participants can see how quickly PCs capture total variation.
axes[1].plot(
    range(10),
    variance_df["cumulative_variance_explained"].head(10),
    color="#222222",
    marker="o",
    label="Cumulative",
)
axes[1].set_title("PCA explained variance")
axes[1].set_ylabel("Proportion of variance")
axes[1].set_ylim(0, 1.05)
axes[1].legend(frameon=False)

# Save the combined PCA figure.
fig.tight_layout()
fig_path = OUTPUT_DIR / "pca_top_canopy_scores_and_variance.png"
fig.savefig(fig_path, dpi=200)
fig_path


## 8. Train/Test Split for Modeling

For the random forest model, PCA is fitted on the training set only. The same scaler and PCA transformation are then applied to the test set. This prevents information from the test samples from influencing the PCs used by the model.


In [ ]:
# Split the data into training and test sets.
# The model learns from the training set and is evaluated on the test set.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_binary,
    test_size=0.25,          # keep 25% of samples for testing
    stratify=y_binary,       # preserve the symptom-class proportions in train and test sets
    random_state=42,         # make the split reproducible
)

# For modeling, fit the scaler only on the training set.
# This avoids leaking information from the test set into preprocessing.
scaler_model = StandardScaler()
X_train_scaled = scaler_model.fit_transform(X_train)
X_test_scaled = scaler_model.transform(X_test)

# Fit PCA only on the training set, then apply that PCA transformation to the test set.
# This is the leakage-safe way to use PCA before model training.
pca_model = PCA(n_components=4, random_state=42)
X_train_pcs = pca_model.fit_transform(X_train_scaled)
X_test_pcs = pca_model.transform(X_test_scaled)

# Give the four principal components readable column names.
pc_cols = ["PC1", "PC2", "PC3", "PC4"]

# Convert arrays back to DataFrames so outputs are easier to inspect.
X_train_pcs = pd.DataFrame(X_train_pcs, columns=pc_cols, index=X_train.index)
X_test_pcs = pd.DataFrame(X_test_pcs, columns=pc_cols, index=X_test.index)

# Print sample counts and class counts so we know what the model is learning from and testing on.
print(f"Training samples: {len(X_train_pcs)}")
print(f"Test samples: {len(X_test_pcs)}")
print("Training labels:")
display(y_train.map({0: "asymptomatic", 1: "symptomatic"}).value_counts())
print("Test labels:")
display(y_test.map({0: "asymptomatic", 1: "symptomatic"}).value_counts())

# Show how much training-set variance is explained by each PC used in the model.
print("Training-set PCA variance explained:")
for pc, var in zip(pc_cols, pca_model.explained_variance_ratio_):
    print(f"{pc}: {var * 100:.1f}%")


## 9. Train a Random Forest Using PC1-PC4

This model asks whether the first four principal components contain enough information to classify symptomatic vs asymptomatic samples.


In [ ]:
# We use a Random Forest here as one example of a supervised machine-learning model for classification,
# but other models may work better depending on the structure, size, and noise characteristics of your dataset.
# In practice, it is often useful to compare multiple approaches (for example, Random Forest, logistic regression,
# support vector machines, or gradient boosting) and evaluate which performs best and why.


# Create the random forest classifier.
# n_estimators=500 means the forest contains 500 decision trees.
# class_weight="balanced" helps when the number of symptomatic and asymptomatic samples differs.
# random_state=42 makes the model reproducible.
# n_jobs=-1 uses all available CPU cores to speed up training.
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

# Train the model using only PC1-PC4 from the training set.
rf.fit(X_train_pcs, y_train)

# Predict hard class labels for the test set: 0 = asymptomatic, 1 = symptomatic.
y_pred = rf.predict(X_test_pcs)

# Predict the probability that each test sample is symptomatic.
# These probabilities are used for AUC.
y_prob = rf.predict_proba(X_test_pcs)[:, 1]

# Collect model performance metrics in a small table.
metrics = pd.DataFrame([{
    "model": "Random Forest using PC1-PC4",
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "symptomatic_recall": recall_score(y_test, y_pred, pos_label=1),
    "asymptomatic_recall": recall_score(y_test, y_pred, pos_label=0),
    "symptomatic_precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
    "symptomatic_f1": f1_score(y_test, y_pred, pos_label=1),
    "auc": roc_auc_score(y_test, y_prob),
}])

# Feature importance tells us which PCs the random forest used most strongly.
pc_importance = pd.DataFrame({
    "PC": pc_cols,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

# Save sample-level predictions so participants can inspect which samples were misclassified.
predictions = pd.DataFrame({
    "actual": y_test.map({0: "asymptomatic", 1: "symptomatic"}),
    "predicted": pd.Series(y_pred, index=y_test.index).map({0: "asymptomatic", 1: "symptomatic"}),
    "prob_symptomatic": y_prob,
}, index=y_test.index)

# Save model outputs to CSV files in the outputs folder.
metrics.to_csv(OUTPUT_DIR / "rf_pc_model_metrics.csv", index=False)
pc_importance.to_csv(OUTPUT_DIR / "rf_pc_feature_importance.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "rf_pc_predictions.csv", index=False)

# Display the main results in the notebook.
display(metrics)
display(pc_importance)


In [ ]:
# AI Prompt: Ask AI to check for overfitting risks
#
# Once participants have seen the train/test split, PCA workflow, and model metrics,
# AI can help them think critically about whether the results are trustworthy.
#
# Try this prompt:
# "Review this top-canopy hyperspectral workflow that uses a train/test split, PCA on the training set,
# and a random forest on PC1-PC4. What are the main overfitting, leakage, or confounding risks
# I should check before trusting the results?"


## 10. Confusion Matrix

For disease detection, do not rely on accuracy alone. Ask which mistakes matter most. In early disease detection, false negatives may be especially costly because they are missed infections.


In [ ]:
# Build a confusion matrix comparing the true test labels with the model predictions.
# Rows are actual labels; columns are predicted labels.
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

# Create a figure for the confusion matrix.
fig, ax = plt.subplots(figsize=(5.5, 4.8))

# ConfusionMatrixDisplay formats the confusion matrix as a labeled plot.
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["asymptomatic", "symptomatic"],
)

# Plot the confusion matrix using a blue color scale.
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Random forest using PC1-PC4")

# Workshop debugging exercise:
# the next line intentionally contains one extra closing parenthesis.
# Run the cell, read the error message, and use the AI prompt below to practice debugging.
fig.tight_layout())

# Save the confusion matrix plot after the typo above is fixed.
fig_path = OUTPUT_DIR / "rf_pc_confusion_matrix.png"
fig.savefig(fig_path, dpi=200)
fig_path


In [ ]:
# AI Prompt: Ask AI to debug an error message
#
# This confusion-matrix cell includes one small intentional typo for workshop practice.
# Run it, read the traceback, and then ask AI for the smallest fix.
#
# Try this prompt:
# "I ran this confusion-matrix plotting cell and got the error below.
# Explain what the error message means and suggest the smallest fix."


## 11. Interpret the Results

Prompts to guide discussion:

1. Do symptomatic and asymptomatic samples separate in PCA space?
2. How much variance do PC1-PC4 explain?
3. Which class does the random forest detect better?